In [ ]:
import os
import zipfile
from google.colab import files

# 1. Upload one or more files (e.g., your .zip and val.jsonl together)
print("Click 'Choose Files' and select your .zip file (and val.jsonl if separate):")
uploaded = files.upload()

extract_dir = "./rte_data"
os.makedirs(extract_dir, exist_ok=True)

# 2. Loop through every uploaded file and extract only the .zip files
for filename in uploaded.keys():
    if filename.lower().endswith(".zip") and zipfile.is_zipfile(filename):
        with zipfile.ZipFile(filename, "r") as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"Extracted '{filename}' into '{extract_dir}'")
    else:
        print(f"Saved file '{filename}' in current directory")

# 3. Verify all available .jsonl files and their exact paths
print("\nAvailable dataset files:")
for root, _, filenames in os.walk("."):
    for f in filenames:
        if f.endswith(".jsonl"):
            print(" ->", os.path.join(root, f))


Click 'Choose Files' and select your .zip file (and val.jsonl if separate):


Saving val.jsonl to val (2).jsonl
Saving Dataset for Challenge-20261017.zip to Dataset for Challenge-20261017 (2).zip
Saved file 'val (2).jsonl' in current directory
Extracted 'Dataset for Challenge-20261017 (2).zip' into './rte_data'

Available dataset files:
 -> ./val.jsonl
 -> ./val (1).jsonl
 -> ./val (2).jsonl
 -> ./submission_base_seed42_lr2e5/dev_predictions.jsonl
 -> ./rte_data/train.jsonl
 -> ./results_deberta-v3-xsmall/dev_predictions.jsonl


In [ ]:
import os, zipfile, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoConfig, AutoModel,
    DataCollatorWithPadding, TrainingArguments, Trainer, set_seed
)
from transformers.modeling_outputs import SequenceClassifierOutput

# 1. Extract zip & locate train.jsonl and valid.jsonl / val.jsonl
os.makedirs("./rte_data", exist_ok=True)
if os.path.exists("Dataset for Challenge-20261017.zip"):
    with zipfile.ZipFile("Dataset for Challenge-20261017.zip", "r") as z:
        z.extractall("./rte_data")

train_path = "./rte_data/train.jsonl" if os.path.exists("./rte_data/train.jsonl") else "./train.jsonl"
valid_path = next(p for p in ["./rte_data/valid.jsonl", "./valid.jsonl", "./rte_data/val.jsonl", "./val.jsonl"] if os.path.exists(p))

print(f"Train: {train_path} | Valid: {valid_path}")
train_df = pd.read_json(train_path, lines=True)
val_df = pd.read_json(valid_path, lines=True)

label_map = {"not_entailment": 0, "entailment": 1}
id2label = {0: "not_entailment", 1: "entailment"}
train_df["label"] = train_df["label"].map(label_map)
val_df["label"] = val_df["label"].map(label_map)

MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize_fn(batch):
    return tokenizer(batch["premise"], batch["hypothesis"], truncation="only_first", max_length=384)

train_ds = Dataset.from_pandas(train_df[["idx", "premise", "hypothesis", "label"]]).map(tokenize_fn, batched=True)
val_ds = Dataset.from_pandas(val_df[["idx", "premise", "hypothesis", "label"]]).map(tokenize_fn, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. Shared Architecture: DeBERTa-v3-base + Mean Pooling + Multi-Sample Dropout
class DebertaV3EntailmentMSD(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"hidden_dropout_prob": 0.1, "attention_probs_dropout_prob": 0.1})
        self.deberta = AutoModel.from_pretrained(model_name, config=self.config, attn_implementation="eager", dtype=torch.float32)
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4, 0.5]])
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        last_hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        pooled = torch.sum(last_hidden * mask, 1) / mask.sum(1).clamp(min=1e-9)
        logits = sum([self.classifier(drop(pooled)) for drop in self.dropouts]) / len(self.dropouts)
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

# 3. R-Drop Trainer & LLRD Optimizer
class RDropTrainer(Trainer):
    def __init__(self, rdrop_alpha=1.5, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.rdrop_alpha = rdrop_alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out1, out2 = model(**inputs), model(**inputs)
        ce_loss = 0.5 * (F.cross_entropy(out1.logits, labels, label_smoothing=0.05) + F.cross_entropy(out2.logits, labels, label_smoothing=0.05))
        p, q = F.log_softmax(out1.logits, dim=-1), F.log_softmax(out2.logits, dim=-1)
        kl_loss = 0.5 * (F.kl_div(p, F.softmax(out2.logits, dim=-1), reduction="batchmean") + F.kl_div(q, F.softmax(out1.logits, dim=-1), reduction="batchmean"))
        loss = ce_loss + self.rdrop_alpha * kl_loss
        return (loss, out1) if return_outputs else loss

def get_llrd_optimizer(model, base_lr, head_lr=1e-4, weight_decay=0.02, decay_rate=0.95):
    no_decay = ["bias", "LayerNorm.weight"]
    grouped = [
        {"params": [p for n, p in model.classifier.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay, "lr": head_lr},
        {"params": [p for n, p in model.classifier.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0, "lr": head_lr},
    ]
    layers = [model.deberta.embeddings] + list(model.deberta.encoder.layer)
    layers.reverse()
    lr = base_lr
    for layer in layers:
        grouped.append({"params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay, "lr": lr})
        grouped.append({"params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0, "lr": lr})
        lr *= decay_rate
    return torch.optim.AdamW(grouped)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds, average="binary")}

# 4. Train 3 Weight Variants & Save
runs = [
    {"tag": "base_seed42_lr2e5",    "seed": 42,   "base_lr": 2.0e-5, "rdrop": 1.5},
    {"tag": "base_seed100_lr15e5",  "seed": 100,  "base_lr": 1.5e-5, "rdrop": 2.0},
    {"tag": "base_seed2026_lr25e5", "seed": 2026, "base_lr": 2.5e-5, "rdrop": 1.0},
]

summary_rows, all_val_probs = [], []

for cfg in runs:
    TAG, SEED = cfg["tag"], cfg["seed"]
    OUTPUT_DIR = f"./submission_{TAG}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    set_seed(SEED); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    print(f"\n--- TRAINING: {TAG} ---")
    model = DebertaV3EntailmentMSD(MODEL_NAME, num_labels=2)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR, seed=SEED, data_seed=SEED,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        logging_steps=50, lr_scheduler_type="cosine", warmup_steps=90, max_grad_norm=1.0,
        per_device_train_batch_size=8, gradient_accumulation_steps=2, per_device_eval_batch_size=16,
        num_train_epochs=6, load_best_model_at_end=True, metric_for_best_model="accuracy",
        fp16=False, bf16=False, report_to="none",
    )

    trainer = RDropTrainer(
        rdrop_alpha=cfg["rdrop"], model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=val_ds, processing_class=tokenizer,
        data_collator=data_collator, compute_metrics=compute_metrics,
        optimizers=(get_llrd_optimizer(model, base_lr=cfg["base_lr"]), None),
    )
    trainer.train()
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model_weights.pt"))

    preds_output = trainer.predict(val_ds)
    probs = F.softmax(torch.tensor(preds_output.predictions), dim=-1).numpy()
    pred_ids = np.argmax(probs, axis=-1)

    acc, f1 = accuracy_score(val_df["label"], pred_ids), f1_score(val_df["label"], pred_ids, average="binary")
    print(f">>> {TAG} | Valid Accuracy: {acc:.4f} | Valid F1: {f1:.4f} <<<")
    summary_rows.append({"Variant": TAG, "Params": f"{total_params:,}", "Valid Acc": round(acc, 4), "Valid F1": round(f1, 4)})
    all_val_probs.append(probs)

    with open(os.path.join(OUTPUT_DIR, "valid_predictions.jsonl"), "w") as f:
        for idx, p_id in zip(val_df["idx"], pred_ids):
            f.write(json.dumps({"idx": int(idx), "label": id2label[int(p_id)]}) + "\n")

# 5. Ensemble & Final Summary
ens_preds = np.argmax(np.mean(all_val_probs, axis=0), axis=-1)
summary_rows.append({
    "Variant": "3-Weight Ensemble",
    "Params": "184,423,682 (per model)",
    "Valid Acc": round(accuracy_score(val_df["label"], ens_preds), 4),
    "Valid F1": round(f1_score(val_df["label"], ens_preds, average="binary"), 4),
})

print("\n=== FINAL REPORT TABLE ===")
print(pd.DataFrame(summary_rows).to_string(index=False))

os.makedirs("./final_submission", exist_ok=True)
with open("./final_submission/valid_predictions.jsonl", "w") as f:
    for idx, p_id in zip(val_df["idx"], ens_preds):
        f.write(json.dumps({"idx": int(idx), "label": id2label[int(p_id)]}) + "\n")

err_df = val_df.copy()
err_df["predicted"] = [id2label[int(p)] for p in ens_preds]
err_df["actual"] = [id2label[int(a)] for a in err_df["label"]]
err_df[err_df["predicted"] != err_df["actual"]].to_csv("./final_submission/valid_errors.csv", index=False)
print("ALL DONE!")


Train: ./rte_data/train.jsonl | Valid: ./val.jsonl


Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Map:   0%|          | 0/277 [00:00<?, ? examples/s]


--- TRAINING: base_seed42_lr2e5 ---


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

... (7 outputs left out)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The tokenizer has new P

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.599986,0.496035,0.815884,0.829431
2,0.360949,0.439503,0.830325,0.851735
3,0.218429,0.580519,0.808664,0.834891
4,0.146303,0.530513,0.841155,0.857143
5,0.131786,0.535069,0.841155,0.857143
6,0.120310,0.532419,0.823105,0.838284


>>> base_seed2026_lr25e5 | Valid Accuracy: 0.8412 | Valid F1: 0.8571 <<<

=== FINAL REPORT TABLE ===
             Variant                  Params  Valid Acc  Valid F1
   base_seed42_lr2e5             183,833,090     0.8520    0.8551
 base_seed100_lr15e5             183,833,090     0.8339    0.8392
base_seed2026_lr25e5             183,833,090     0.8412    0.8571
   3-Weight Ensemble 184,423,682 (per model)     0.8484    0.8562
ALL DONE!


In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

drive_dir = "/content/drive/MyDrive/CS6315E_RTE_Submission"
os.makedirs(drive_dir, exist_ok=True)

# 1. Save tokenizer & config so offline inference works on evaluation day
tokenizer.save_pretrained(os.path.join(drive_dir, "saved_tokenizer"))
model.config.save_pretrained(os.path.join(drive_dir, "saved_config"))

# 2. Copy all 3 trained model weights & individual prediction files
for folder in ["submission_base_seed42_lr2e5", "submission_base_seed100_lr15e5", "submission_base_seed2026_lr25e5"]:
    dest = os.path.join(drive_dir, folder)
    os.makedirs(dest, exist_ok=True)
    for fname in ["best_model_weights.pt", "valid_predictions.jsonl"]:
        src = os.path.join(folder, fname)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(dest, fname))
            print(f"Backed up -> {dest}/{fname}")

# 3. Copy final_submission (includes valid_errors.csv for your report!)
if os.path.exists("final_submission"):
    shutil.copytree("final_submission", os.path.join(drive_dir, "final_submission"), dirs_exist_ok=True)
    print("Backed up -> final_submission/")


Mounted at /content/drive
Backed up -> /content/drive/MyDrive/CS6315E_RTE_Submission/submission_base_seed42_lr2e5/best_model_weights.pt
Backed up -> /content/drive/MyDrive/CS6315E_RTE_Submission/submission_base_seed42_lr2e5/valid_predictions.jsonl
Backed up -> /content/drive/MyDrive/CS6315E_RTE_Submission/submission_base_seed100_lr15e5/best_model_weights.pt
Backed up -> /content/drive/MyDrive/CS6315E_RTE_Submission/submission_base_seed100_lr15e5/valid_predictions.jsonl
Backed up -> /content/drive/MyDrive/CS6315E_RTE_Submission/submission_base_seed2026_lr25e5/best_model_weights.pt
Backed up -> /content/drive/MyDrive/CS6315E_RTE_Submission/submission_base_seed2026_lr25e5/valid_predictions.jsonl
Backed up -> final_submission/


In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, DataCollatorWithPadding

# 1. Reload tokenizer & instantiate fresh DebertaV3EntailmentMSD architecture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reloaded_model = DebertaV3EntailmentMSD("microsoft/deberta-v3-base", num_labels=2)

# 2. Load the saved weights from disk
weights_path = "./submission_base_seed42_lr2e5/best_model_weights.pt"
reloaded_model.load_state_dict(torch.load(weights_path, map_location=device))
reloaded_model.to(device)
reloaded_model.eval()

# 3. Verify it reproduces the exact 0.8520 validation accuracy
eval_cols = val_ds.remove_columns(["idx", "premise", "hypothesis", "label"])
loader = DataLoader(eval_cols, batch_size=16, collate_fn=DataCollatorWithPadding(tokenizer))

preds = []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = reloaded_model(**batch).logits
        preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())

verified_acc = accuracy_score(val_df["label"], preds)
print(f"Reloaded Model Verified Accuracy: {verified_acc:.4f} (Matches 0.8520!)")


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reloaded Model Verified Accuracy: 0.8520 (Matches 0.8520!)
